<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# NB03: Data Integration

Assembles the integrated species-level dataset from five BERDL tables:
- `pangenome` + `gtdb_species_clade` — pangenome metrics and taxonomy
- `genome` — genome-to-species mapping
- `gapmind_pathways` — metabolic pathway completeness
- `ncbi_env` — environment metadata for niche breadth and lifestyle classification
- `alphaearth_embeddings_all_years` — structural embedding coverage

**Outputs**: `data/species_integrated.csv` (all species), `data/species_integrated_10plus.csv` (>=10 genomes)

**Requires**: Spark (run on BERDL JupyterHub). Approx 15 min runtime.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
get_spark_session()

RuntimeError: Spark Connect session did not respond within 90s: the server is listening but not serving sessions (likely a half-broken/zombie driver). Restart it with start_spark_connect_server(force_restart=True); if this happens in an existing kernel, restart the kernel first. See docs/jupyterhub-active-users.md §9.6.

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

# Spark session is pre-initialized in the JupyterHub kernel.
# Do NOT import get_spark_session on-cluster.
print(f'Spark version: {spark.version}')

## 1. Pangenome metrics and taxonomy

Join `pangenome` with `gtdb_species_clade` to get species-level pangenome
statistics and compute the openness index.

**Openness** = (no_aux_genome + no_singleton_gene_clusters) / no_gene_clusters.
Note: `no_aux_genome` counts individual auxiliary genes while `no_gene_clusters`
counts gene cluster families, so this ratio can exceed 1.0 for highly open pangenomes.
It is treated as a relative ranking, not a bounded proportion.

In [ ]:
pangenome_df = spark.sql("""
SELECT
    p.gtdb_species_clade_id AS species_id,
    p.no_genomes,
    p.no_core,
    p.no_aux_genome AS no_aux,
    p.no_singleton_gene_clusters AS no_singleton,
    p.no_gene_clusters,
    p.no_CDSes,
    (p.no_aux_genome + p.no_singleton_gene_clusters) / p.no_gene_clusters AS openness
FROM kbase_ke_pangenome.pangenome p
""").toPandas()

# Extract taxonomy from GTDB species name
taxonomy_df = spark.sql("""
SELECT
    s.gtdb_species_clade_id AS species_id,
    s.GTDB_species,
    REGEXP_EXTRACT(s.GTDB_species, 'p__(\\\\w+)', 1) AS phylum,
    REGEXP_EXTRACT(s.GTDB_species, 'c__(\\\\w+)', 1) AS class,
    REGEXP_EXTRACT(s.GTDB_species, 'o__(\\\\w+)', 1) AS `order`,
    REGEXP_EXTRACT(s.GTDB_species, 'f__(\\\\w+)', 1) AS family,
    REGEXP_EXTRACT(s.GTDB_species, 'g__(\\\\w+)', 1) AS genus
FROM kbase_ke_pangenome.gtdb_species_clade s
""").toPandas()

# If REGEX extraction doesn't work, fall back to the genome table taxonomy
if taxonomy_df['phylum'].isna().all():
    taxonomy_df = spark.sql("""
    SELECT DISTINCT
        g.gtdb_species_clade_id AS species_id,
        t.phylum, t.class, t.`order`, t.family, t.genus
    FROM kbase_ke_pangenome.genome g
    JOIN kbase_ke_pangenome.gtdb_taxonomy_r214v1 t ON g.genome_id = t.genome_id
    """).toPandas()

pangenome_with_tax = pangenome_df.merge(taxonomy_df, on='species_id', how='left')
print(f'Pangenome metrics: {len(pangenome_with_tax)} species')
print(pangenome_with_tax[['openness']].describe())

## 2. GapMind pathway completeness per species

Aggregate pathway scores to species level. 

**PITFALL (docs/pitfalls.md)**: `score_simplified` is binary 0.0/1.0 (pathway
complete vs incomplete), NOT categorical strings like 'High'/'Medium'/'Low'.
Use numeric comparisons only.

In [ ]:
# Count complete pathways per genome, then aggregate to species
pathway_df = spark.sql("""
WITH genome_pathway_counts AS (
    SELECT
        g.gtdb_species_clade_id AS species_id,
        gp.genome_id,
        SUM(CASE WHEN CAST(gp.score_simplified AS DOUBLE) = 1.0 THEN 1 ELSE 0 END) AS complete_pathways,
        CAST(AVG(gp.score) AS DOUBLE) AS mean_score,
        COUNT(*) AS pathways_assessed
    FROM kbase_ke_pangenome.genome g
    JOIN kbase_ke_pangenome.gapmind_pathways gp ON g.genome_id = gp.genome_id
    GROUP BY g.gtdb_species_clade_id, gp.genome_id
)
SELECT
    species_id,
    COUNT(*) AS n_genomes_with_pathways,
    ROUND(AVG(complete_pathways), 2) AS mean_complete_pathways,
    ROUND(STDDEV(complete_pathways), 2) AS std_complete_pathways,
    ROUND(AVG(mean_score), 3) AS mean_pathway_score,
    ROUND(AVG(pathways_assessed), 1) AS mean_pathways_assessed,
    ROUND(
        CASE WHEN AVG(complete_pathways) > 0
             THEN STDDEV(complete_pathways) / AVG(complete_pathways)
             ELSE 0 END,
        3
    ) AS pathway_cv
FROM genome_pathway_counts
GROUP BY species_id
""").toPandas()

print(f'Pathway data: {len(pathway_df)} species')
print(pathway_df[['mean_complete_pathways', 'mean_pathway_score']].describe())

## 3. Environment classification and niche breadth

### Derivation

**`niche_breadth`**: For each species, we count the fraction of genomes in each
environment category (derived from NCBI BioSample `isolation_source` and `host`
fields). Niche breadth is the normalized Shannon diversity:
`H / ln(n_categories)`, giving a 0-1 measure of environmental spread. Species
whose genomes span both host and free-living environments get higher niche breadth.

**`environment_type`**: Majority-vote classification for each species:
- `host_associated`: >50% of genomes from host/clinical/animal sources
- `free_living`: >50% of genomes from soil/water/sediment/plant-surface sources
- Empty if no environment data or ambiguous

Environment keywords are derived from NCBI BioSample `harmonized_name = 'isolation_source'`
and `harmonized_name = 'host'` entries in `kbase_ke_pangenome.ncbi_env`.

In [ ]:
# Get environment annotations per genome
env_raw = spark.sql("""
SELECT
    g.gtdb_species_clade_id AS species_id,
    g.genome_id,
    ne.harmonized_name,
    ne.value
FROM kbase_ke_pangenome.genome g
JOIN kbase_ke_pangenome.ncbi_env ne ON g.genome_id = ne.genome_id
WHERE ne.harmonized_name IN ('isolation_source', 'host', 'geo_loc_name')
""").toPandas()

print(f'Environment annotations: {len(env_raw)} records for {env_raw["genome_id"].nunique()} genomes')

In [ ]:
# Classify each genome as host_associated or free_living
host_keywords = ['human', 'homo sapiens', 'patient', 'clinical', 'blood',
                 'sputum', 'urine', 'wound', 'stool', 'feces', 'fecal',
                 'gut', 'intestin', 'oral', 'skin', 'respiratory',
                 'animal', 'chicken', 'pig', 'cattle', 'bovine', 'mouse',
                 'insect', 'fish', 'host']
free_keywords = ['soil', 'water', 'marine', 'ocean', 'river', 'lake',
                 'sediment', 'rhizosphere', 'wastewater', 'sewage',
                 'air', 'environment', 'food', 'plant', 'leaf', 'root']

def classify_genome(group):
    text = ' '.join(group['value'].dropna().str.lower())
    host_score = sum(1 for kw in host_keywords if kw in text)
    free_score = sum(1 for kw in free_keywords if kw in text)
    if host_score > free_score:
        return 'host_associated'
    elif free_score > host_score:
        return 'free_living'
    return 'unknown'

genome_env = env_raw.groupby('genome_id').apply(classify_genome).reset_index()
genome_env.columns = ['genome_id', 'env_class']
genome_env = genome_env[genome_env['env_class'] != 'unknown']

# Add species_id back
genome_env = genome_env.merge(
    env_raw[['genome_id', 'species_id']].drop_duplicates(),
    on='genome_id'
)

print(f'Classified genomes: {len(genome_env)}')
print(genome_env['env_class'].value_counts())

In [ ]:
# Compute niche_breadth (normalized Shannon diversity) and environment_type (majority vote)
from collections import Counter

def compute_niche_metrics(group):
    counts = Counter(group['env_class'])
    total = sum(counts.values())
    n_categories = len(counts)
    
    # Shannon diversity
    if n_categories <= 1 or total == 0:
        niche_breadth = 0.0
    else:
        probs = [c / total for c in counts.values()]
        H = -sum(p * np.log(p) for p in probs if p > 0)
        niche_breadth = H / np.log(n_categories)
    
    # Majority vote
    majority = counts.most_common(1)[0][0]
    majority_frac = counts[majority] / total
    env_type = majority if majority_frac > 0.5 else ''
    
    return pd.Series({
        'niche_breadth': round(niche_breadth, 4),
        'environment_type': env_type
    })

species_env = genome_env.groupby('species_id').apply(compute_niche_metrics).reset_index()
print(f'Species with environment data: {len(species_env)}')
print(species_env['environment_type'].value_counts())

## 4. AlphaEarth embedding coverage

In [ ]:
ae_coverage = spark.sql("""
SELECT
    g.gtdb_species_clade_id AS species_id,
    COUNT(DISTINCT ae.genome_id) AS n_genomes_ae
FROM kbase_ke_pangenome.genome g
JOIN kbase_ke_pangenome.alphaearth_embeddings_all_years ae ON g.genome_id = ae.genome_id
GROUP BY g.gtdb_species_clade_id
""").toPandas()

print(f'Species with AlphaEarth embeddings: {len(ae_coverage)}')

## 5. Integrate and save

In [ ]:
# Merge all DataFrames
integrated = pangenome_with_tax.merge(pathway_df, on='species_id', how='left')
integrated = integrated.merge(species_env, on='species_id', how='left')
integrated = integrated.merge(ae_coverage, on='species_id', how='left')

# Drop the GTDB_species column if present (came from taxonomy join)
if 'GTDB_species' in integrated.columns:
    integrated = integrated.drop(columns=['GTDB_species'])

# Sort by no_genomes descending
integrated = integrated.sort_values('no_genomes', ascending=False)

print(f'Integrated dataset: {integrated.shape}')
print(f'Columns: {list(integrated.columns)}')
print(f'\nCoverage:')
print(f'  Total species: {len(integrated)}')
print(f'  With pathway data: {integrated["n_genomes_with_pathways"].notna().sum()}')
print(f'  With environment data: {integrated["niche_breadth"].notna().sum()}')
print(f'  With AlphaEarth: {integrated["n_genomes_ae"].notna().sum()}')

# Save full dataset
output_full = os.path.join(DATA_DIR, 'species_integrated.csv')
integrated.to_csv(output_full, index=False)
print(f'\nSaved: {output_full} ({len(integrated)} rows)')

# Filter to species with >= 10 genomes
integrated_10 = integrated[integrated['no_genomes'] >= 10].copy()
output_10 = os.path.join(DATA_DIR, 'species_integrated_10plus.csv')
integrated_10.to_csv(output_10, index=False)
print(f'Saved: {output_10} ({len(integrated_10)} rows)')

## Summary

| Output | Rows | Description |
|--------|------|-------------|
| `data/species_integrated.csv` | ~27,690 | Full integrated dataset |
| `data/species_integrated_10plus.csv` | ~2,812 | Filtered to species with >= 10 genomes |

**Next**: Run `04_statistical_analysis.ipynb` (no Spark needed — reads from these CSVs).